# 02 ? Normalize the full CAD log
Run all cells from the repository root or `notebooks/`. Preserves every source row, call number, address, date/time, latitude and longitude. No filtering by study dates, initiation or geography; no call deduplication or parcel assignment.

Uses the shared [Publication 28 formatting rules](https://pe.usps.com/text/pub28/welcome.htm) from notebook 01. Street-only CAD addresses do not receive an invented Savannah locality or ZIP. Street candidate keys need subsequent coordinate/locality checks before any match. Intersections, blocks, landmarks, missing addresses, and unsupported forms enter review and have blank candidate keys. Unresolved text is retained. Coordinates are checked only for numeric, finite, global range and zero-pair validity; valid ranges do not establish positional accuracy. Source date/time fields are preserved alongside derived timestamps with no assumed timezone.

No USPS delivery validation. Outputs are local working data, not linked from the public website. Dependencies: `pip install -r notebooks/requirements.txt`.

In [1]:
from pathlib import Path
import sys, json, hashlib, platform
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'raw_data').is_dir(): ROOT = ROOT.parent
assert (ROOT / 'raw_data').is_dir()
sys.path.insert(0, str(ROOT))
from processing.cad_normalization import normalize_cad_address, VERSION
INPUT = ROOT / 'raw_data' / 'CAD-log.xlsx'
SHEET = 'Call For Service Basic'
ADDRESS_COLUMN = 'Address'
OUTPUT = ROOT / 'processed_data' / 'cad'
OUTPUT.mkdir(parents=True, exist_ok=True)
source_hash = hashlib.sha256(INPUT.read_bytes()).hexdigest()
source = pd.read_excel(INPUT, sheet_name=SHEET, dtype=str).fillna('')
assert {'Call Number', ADDRESS_COLUMN, 'Latitude', 'Longitude', 'Date', 'Time'}.issubset(source.columns)
print(f'{len(source):,} source rows loaded')
display(source.head(3))

146,213 source rows loaded


,Precinct,CRN,Call Number,Day of Week,Date,Time,Hour,Call Type,Priority Desc,Address,Latitude,Longitude,Beat,Initiation
0,Southside,SPD240101004,24001000002,Monday,01/01/2024,00:01,00,53 SHOTS FIRED(heard only),P5 Immediate Call,13 Ventura Blvd,31.985909,-81.13918,46,Cit Initiated
1,Southside,,24001000003,Monday,01/01/2024,00:02,00,53 SHOTS FIRED(heard only),P5 Immediate Call,W Fairmont Ave / White Bluff Rd,32.000297,-81.121552,42,Cit Initiated
2,Southside,,24001000005,Monday,01/01/2024,00:03,00,53 SHOTS FIRED(heard only),P5 Immediate Call,202 Travis St,31.996756,-81.118493,43,Cit Initiated


In [2]:
assert normalize_cad_address('13 Ventura Boulevard')['street_building_candidate_key'] == '13 VENTURA BLVD'
assert not normalize_cad_address('13 Ventura Blvd')['city']
assert normalize_cad_address('W Fairmont Ave / White Bluff Rd')['review_required']
assert normalize_cad_address('100 BLOCK E BAY ST')['street_building_candidate_key'] == ''
assert normalize_cad_address('202 Travis St')['review_required'] is False
assert normalize_cad_address('')['review_required']
assert normalize_cad_address('101 E Gaston St #8B')['unit_identifier'] == '8B'
print('CAD rule checks passed')
# Normalize unique addresses once, then expand back to all source rows.
lookup = {a: normalize_cad_address(a) for a in source[ADDRESS_COLUMN].unique()}
normalized = pd.DataFrame([lookup[a] for a in source[ADDRESS_COLUMN]])
result = pd.concat([source.reset_index(drop=True), normalized], axis=1)
result.insert(0, 'source_excel_row', range(2, len(result)+2))
result.insert(0, 'source_sheet', SHEET)
result.insert(0, 'source_file', INPUT.name)
result['latitude_numeric'] = pd.to_numeric(source.Latitude, errors='coerce')
result['longitude_numeric'] = pd.to_numeric(source.Longitude, errors='coerce')
result['coordinates_valid_global_range'] = (result.latitude_numeric.between(-90,90) & result.longitude_numeric.between(-180,180) & ~(result.latitude_numeric.eq(0) & result.longitude_numeric.eq(0)))
result['call_timestamp_local_unverified'] = pd.to_datetime(source.Date + ' ' + source.Time, format='%m/%d/%Y %H:%M', errors='coerce')
result['duplicate_call_number'] = source['Call Number'].ne('') & source['Call Number'].duplicated(keep=False)
result['duplicate_source_row'] = source.duplicated(keep=False)
result['record_review_required'] = result.review_required | ~result.coordinates_valid_global_range | result.call_timestamp_local_unverified.isna() | source['Call Number'].eq('')
assert len(result) == len(source)
for c in source.columns: assert result[c].equals(source[c]), c
assert not result.usps_delivery_validated.any()
assert result.loc[result.review_required, 'street_building_candidate_key'].eq('').all()
display(result.address_kind.value_counts())
display(result.loc[result.review_required, ['Address','address_kind','review_reasons']].drop_duplicates().head(25))

CAD rule checks passed


address_kind
street_address                             91265
intersection_or_location_description       32628
house_number_range_or_hyphenated_number    12486
street_or_landmark                          9643
block_location                               187
missing                                        4
Name: count, dtype: int64

,Address,address_kind,review_reasons
1,W Fairmont Ave / White Bluff Rd,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
6,W Bay St / Fahm St,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
12,W BAY ST,street_or_landmark,unparsed_street_address; street_or_landmark
14,Audubon Dr / Eleanor St,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
16,Turpentine Trl / Skylark Rd,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
17,Louisiana Ave / North Carolina Ave,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
19,Montgomery St / W Derenne Ave,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
20,gltigny st/staley ave,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
21,Bull St / E Hull St,intersection_or_location_description,unparsed_street_address; intersection_or_locat...
26,BULL ST / W 53RD ST,intersection_or_location_description,unparsed_street_address; intersection_or_locat...


In [3]:
result.to_csv(OUTPUT / 'cad_normalized.csv', index=False)
result.loc[result.record_review_required].to_csv(OUTPUT / 'cad_record_review.csv', index=False)
address_audit = pd.DataFrame(lookup.values())
address_audit['source_row_count'] = address_audit.address_original.map(source[ADDRESS_COLUMN].value_counts())
address_audit.to_csv(OUTPUT / 'cad_unique_addresses.csv', index=False)
address_audit.loc[address_audit.review_required].to_csv(OUTPUT / 'cad_address_review.csv', index=False)
result.loc[result.duplicate_call_number | result.duplicate_source_row].to_csv(OUTPUT / 'cad_duplicate_records.csv', index=False)
summary = {
 'input_file': str(INPUT.relative_to(ROOT)), 'input_sha256': source_hash, 'source_sheet': SHEET,
 'executed_utc': datetime.now(timezone.utc).isoformat(), 'normalization_version': VERSION,
 'code_sha256': {p: hashlib.sha256((ROOT/p).read_bytes()).hexdigest() for p in ['processing/address_normalization.py','processing/cad_normalization.py']},
 'python_version': platform.python_version(), 'pandas_version': pd.__version__,
 'rows': len(result), 'unique_addresses': len(lookup),
 'address_review_rows': int(result.review_required.sum()),
 'record_review_rows': int(result.record_review_required.sum()),
 'invalid_or_missing_coordinates': int((~result.coordinates_valid_global_range).sum()),
 'unparsed_timestamps': int(result.call_timestamp_local_unverified.isna().sum()),
 'missing_call_numbers': int(source['Call Number'].eq('').sum()),
 'duplicate_call_number_rows': int(result.duplicate_call_number.sum()),
 'duplicate_source_rows': int(result.duplicate_source_row.sum()),
 'address_kinds': result.address_kind.value_counts().to_dict(),
 'review_reason_counts': result.loc[result.review_required,'review_reasons'].str.split('; ').explode().value_counts().to_dict(),
 'date_start': str(result.call_timestamp_local_unverified.min()), 'date_end': str(result.call_timestamp_local_unverified.max()),
 'usps_delivery_validated': False,
 'note': 'All rows preserved; street-only candidate keys are not confirmed parcel matches. No timezone or locality inferred.'
}
(OUTPUT/'normalization_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
assert hashlib.sha256(INPUT.read_bytes()).hexdigest() == source_hash
display(summary)
print('Outputs:', OUTPUT)

{'input_file': 'raw_data\\CAD-log.xlsx',
 'input_sha256': '2fab32a09b654fc49bef54418195c235c875221328d0c233bb793c5b278ce5d1',
 'source_sheet': 'Call For Service Basic',
 'executed_utc': '2026-10-27T22:08:04.638182+00:00',
 'normalization_version': '1.0.0',
 'code_sha256': {'processing/address_normalization.py': 'a8008437ba7dcb8171b96f0a759542524cb4079c408cdbe61063fa115a85e546',
  'processing/cad_normalization.py': '48a030eb9e0ae1d1fa35c650df491ba8256a14981c33bfffb111b8da7c4deffb'},
 'python_version': '3.14.6',
 'pandas_version': '3.0.3',
 'rows': 146213,
 'unique_addresses': 40624,
 'address_review_rows': 56963,
 'record_review_rows': 58545,
 'invalid_or_missing_coordinates': 6053,
 'unparsed_timestamps': 4,
 'missing_call_numbers': 3,
 'duplicate_call_number_rows': 670,
 'duplicate_source_rows': 2,
 'address_kinds': {'street_address': 91265,
  'intersection_or_location_description': 32628,
  'house_number_range_or_hyphenated_number': 12486,
  'street_or_landmark': 9643,
  'block_locat

Outputs: C:\Users\Admin\savannah-stvr-cad-study\processed_data\cad
